# Airspace Object Detection: Demo Video

This notebook builds a ~1-minute demo video of the whole system, for the GitHub README and the application:

1. **Real drone video, tracked.** Two Anti-UAV300 test clips the model never saw, with the combined 6-class model + BoT-SORT (camera-motion compensation, hybrid filter). Ground truth shown as a thin white box.
2. **Known classes.** Open Images photos of airplanes, helicopters, birds, kites and parachutes (a different source from the training data), labelled by the combined model.
3. **Things it was never trained on.** Open Images balloons, rockets and missiles. The YOLO-World unknown-object flag marks them **UNKNOWN** in red.
4. An end card with the headline numbers.

The photo segments are a **curated showcase** (only photos where the system got it right are picked). The honest, full-test-set numbers are in the other notebooks and on the end card.

**How to run:** inputs are the combined-model notebook output and the `mm1991/anti-uav-dataset-300` dataset. GPU T4 x2, Internet on, no secrets. Save Version → Save & Run All (about 15 to 25 minutes). Output: `airspace_demo.mp4` (H.264, 1280x720).

**v2 (after v1):** adds airplanes (Open Images calls them "Aircraft" / "Fixed-wing aircraft", so v1 found none) and only shows an unknown-object photo when nothing in it got a known label (v1 showed a balloon photo where a second balloon was labelled "parachute").

**v3 (after v2):** airplanes come only from "Fixed-wing aircraft" (v2's "Aircraft" class let in a blimp, which the model wrongly called an airplane), and every photo must show mostly sky (v2 picked indoor exercise balls tagged "Balloon").

**v4 (after v3):** the sky test counts only blue sky (v3's "bright grey" also matched indoor white walls, so party balloons got in), and no photo is used twice.

**v5 (after v4):** a known-class photo is shown only if every box on it matches a labelled object (v4 showed a skydiver whose face was boxed as "parachute").

In [ ]:
!pip install -q ultralytics lap

In [ ]:
# 1. Models and helpers
import glob, os, random, collections, json, subprocess
import numpy as np, cv2, pandas as pd, requests, concurrent.futures as cf, yaml
from ultralytics import YOLO
from ultralytics.utils import ROOT
random.seed(1)
W = glob.glob("/kaggle/input/**/combined_640/weights/best.pt", recursive=True)[0]
known = YOLO(W)
KNOWN = list(known.names.values())
print("Known classes:", KNOWN)

VOCAB = ["airplane", "bird", "helicopter", "kite", "parachute", "drone",
         "balloon", "hot air balloon", "blimp", "airship", "paraglider", "hang glider",
         "glider", "rocket", "missile", "satellite", "flying insect", "butterfly",
         "plastic bag", "sky lantern", "jet", "fighter jet", "quadcopter"]
SYNONYM = {"jet": "airplane", "fighter jet": "airplane", "quadcopter": "drone", "glider": "airplane"}
world = YOLO("yolov8x-worldv2.pt")
world.set_classes(VOCAB)

def iou(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

def detect(img, known_conf=0.25, world_conf=0.25, match_iou=0.3):
    """Same fusion rule as the unknown-flag notebook (v2)."""
    k = known.predict(img, conf=known_conf, imgsz=640, verbose=False)[0].boxes
    w = world.predict(img, conf=world_conf, imgsz=640, verbose=False)[0].boxes
    kb, wb = k.xyxy.cpu().numpy(), w.xyxy.cpu().numpy()
    out = [dict(box=b, label=KNOWN[int(c)], conf=float(s), unknown=False)
           for b, c, s in zip(kb, k.cls.cpu().numpy(), k.conf.cpu().numpy())]
    ov = iou(wb, kb)
    for i, (b, c, s) in enumerate(zip(wb, w.cls.cpu().numpy(), w.conf.cpu().numpy())):
        hint = VOCAB[int(c)]
        is_known = SYNONYM.get(hint, hint) in KNOWN
        if is_known:
            continue
        if ov.shape[1] == 0 or ov[i].max() < match_iou:
            out.append(dict(box=b, label="UNKNOWN", hint=hint, conf=float(s), unknown=True))
        else:
            j = int(ov[i].argmax())
            if float(s) > out[j]["conf"] and not out[j]["unknown"]:
                out[j] = dict(box=b, label="UNKNOWN", hint=hint, conf=float(s), unknown=True)
    return out

# Video canvas
VW, VH, FPS = 1280, 720, 25
GREEN, RED, WHITE, CYAN = (60, 220, 60), (60, 60, 255), (255, 255, 255), (255, 210, 0)
FONT = cv2.FONT_HERSHEY_DUPLEX
writer = cv2.VideoWriter("/kaggle/working/raw_demo.mp4", cv2.VideoWriter_fourcc(*"mp4v"), FPS, (VW, VH))

def fit(img):
    """Letterbox an image into the 1280x720 canvas. Returns canvas, scale, offset."""
    h, w = img.shape[:2]
    s = min(VW / w, VH / h)
    r = cv2.resize(img, (int(w * s), int(h * s)))
    c = np.full((VH, VW, 3), 18, np.uint8)
    ox, oy = (VW - r.shape[1]) // 2, (VH - r.shape[0]) // 2
    c[oy:oy + r.shape[0], ox:ox + r.shape[1]] = r
    return c, s, ox, oy

def label(img, text, x, y, col, scale=0.75):
    (tw, th), _ = cv2.getTextSize(text, FONT, scale, 2)
    y = max(y, th + 8)
    cv2.rectangle(img, (x, y - th - 8), (x + tw + 8, y + 2), col, -1)
    cv2.putText(img, text, (x + 4, y - 4), FONT, scale, (0, 0, 0), 2, cv2.LINE_AA)

def banner(img, text):
    cv2.rectangle(img, (0, VH - 44), (VW, VH), (0, 0, 0), -1)
    cv2.putText(img, text, (20, VH - 14), FONT, 0.75, WHITE, 1, cv2.LINE_AA)

def card(lines, seconds=3.0):
    """Title card: first line big, the rest smaller."""
    img = np.full((VH, VW, 3), 18, np.uint8)
    y = VH // 2 - 30 * len(lines)
    for i, t in enumerate(lines):
        sc, th = (1.5, 3) if i == 0 else (0.85, 1)
        (tw, _), _ = cv2.getTextSize(t, FONT, sc, th)
        cv2.putText(img, t, ((VW - tw) // 2, y), FONT, sc, CYAN if i == 0 else WHITE, th, cv2.LINE_AA)
        y += 80 if i == 0 else 48
    for _ in range(int(seconds * FPS)):
        writer.write(img)
print("ready")

In [ ]:
# 2. Title card
card(["Airspace Object Detection",
      "Detect everything in the air: 6 known classes + an UNKNOWN flag",
      "YOLO11s trained on AOD4 + MBAOD  |  YOLO-World for unknowns  |  BoT-SORT tracking"], 4)

In [ ]:
# 3. Part 1: tracked drone video (Anti-UAV300, never seen in training)
def load_gt(video):
    d = os.path.dirname(video)
    path = next(p for p in [os.path.join(d, "visible.json")] + glob.glob(os.path.join(d, "*isible*.json")) if os.path.exists(p))
    j = json.load(open(path))
    rects, exist = j["gt_rect"], j.get("exist", [1] * len(j["gt_rect"]))
    return [np.array([r[0], r[1], r[0] + r[2], r[1] + r[3]], float) if e and r and len(r) == 4 and r[2] > 0 and r[3] > 0 else None
            for r, e in zip(rects, exist)]

cfg = yaml.safe_load(open(ROOT / "cfg" / "trackers" / "botsort.yaml"))
cfg.update(track_buffer=60, match_thresh=0.9, gmc_method="sparseOptFlow", with_reid=False)
TRK = "/kaggle/working/botsort_gmc.yaml"; yaml.safe_dump(cfg, open(TRK, "w"))
MIN_TRACK, SURE_CONF, CLIP = 5, 0.5, 250   # 10 s per clip

vids = sorted(glob.glob("/kaggle/input/**/test/**/visible.mp4", recursive=True))
# Quick scan: pick 2 clips where the drone is found in most of the first 100 frames, preferring different scenes
def quick(v):
    gt, cap, hit, n = load_gt(v), cv2.VideoCapture(v), 0, 0
    for f in range(0, 100, 5):
        cap.set(cv2.CAP_PROP_POS_FRAMES, f); ok, img = cap.read()
        if not ok or gt[f] is None: continue
        b = known.predict(img, conf=0.25, verbose=False)[0].boxes.xyxy.cpu().numpy()
        n += 1; hit += bool(len(b)) and iou(gt[f][None], b).max() >= 0.3
    cap.release(); return hit / max(n, 1)
cand = random.sample(vids, min(30, len(vids)))
scores = sorted(((quick(v), v) for v in cand), reverse=True)
demo_vids = [scores[0][1], scores[len(scores) // 4][1]]   # an easy clip and a harder one
print([(round(s, 2), os.path.basename(os.path.dirname(v))) for s, v in scores[:10]])
print("Demo clips:", demo_vids)

card(["Part 1: real drone video",
      "Anti-UAV300 test clips the model has never seen",
      "Green = detection + track ID   |   thin white box = ground truth"], 3)
for vi, v in enumerate(demo_vids):
    gt, model, seen = load_gt(v), YOLO(W), collections.Counter()
    cap = cv2.VideoCapture(v)
    for f in range(CLIP):
        ok, img = cap.read()
        if not ok: break
        r = model.track(img, imgsz=640, conf=0.1, persist=True, tracker=TRK, verbose=False)[0]
        c, s, ox, oy = fit(img)
        if gt[f] is not None:
            x1, y1, x2, y2 = (gt[f] * s + [ox, oy, ox, oy]).astype(int)
            cv2.rectangle(c, (x1 - 3, y1 - 3), (x2 + 3, y2 + 3), WHITE, 1)
        if r.boxes.id is not None:
            for b, cl, i, cf_ in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy(),
                                     r.boxes.id.cpu().numpy(), r.boxes.conf.cpu().numpy()):
                seen[int(i)] += 1
                if seen[int(i)] >= MIN_TRACK or cf_ >= SURE_CONF:
                    x1, y1, x2, y2 = (b * s + [ox, oy, ox, oy]).astype(int)
                    cv2.rectangle(c, (x1, y1), (x2, y2), GREEN, 2)
                    label(c, f"{KNOWN[int(cl)]} #{int(i)} {cf_:.2f}", x1, y1 - 4, GREEN, 0.6)
        banner(c, f"Clip {vi + 1}/2  |  YOLO11s + BoT-SORT (camera-motion comp.)  |  frame {f}")
        writer.write(c)
    cap.release()
print("Part 1 done")

In [ ]:
# 4. Photos from Open Images (a different source from the training data)
BASE = "https://storage.googleapis.com/openimages"
cls = pd.read_csv(f"{BASE}/v7/oidv7-class-descriptions-boxable.csv", header=None, names=["mid", "name"])
KNOWN_OI = {"Airplane": "airplane", "Fixed-wing aircraft": "airplane", "Helicopter": "helicopter", "Bird": "bird", "Kite": "kite", "Parachute": "parachute"}
UNK_OI = ["Balloon", "Rocket", "Missile", "Airship"]
mids = dict(cls[cls.name.isin(list(KNOWN_OI) + UNK_OI)][["mid", "name"]].values)
print("Open Images classes:", sorted(mids.values()))
print("Aircraft-like names available:", [n for n in cls.name if "craft" in n.lower() or "plane" in n.lower()])
ann = pd.read_csv(f"{BASE}/v5/validation-annotations-bbox.csv")
ann = ann[ann.LabelName.isin(mids) & (ann.IsGroupOf == 0)]
ann = ann[((ann.XMax - ann.XMin) * (ann.YMax - ann.YMin)) > 0.01]
ann["cls"] = ann.LabelName.map(mids)
os.makedirs("/kaggle/working/oi", exist_ok=True)
def fetch(i):
    p = f"/kaggle/working/oi/{i}.jpg"
    r = requests.get(f"https://open-images-dataset.s3.amazonaws.com/validation/{i}.jpg", timeout=20)
    if r.ok:
        open(p, "wb").write(r.content); return i
pick = {}
for c in list(KNOWN_OI) + UNK_OI:
    ids = ann[ann.cls == c].ImageID.unique().tolist(); random.shuffle(ids)
    pick[c] = ids[:80]
with cf.ThreadPoolExecutor(16) as ex:
    got = set(i for i in ex.map(fetch, sum(pick.values(), [])) if i)
print(len(got), "photos downloaded")

def gt_boxes(i, c):
    g = ann[(ann.ImageID == i) & (ann.cls == c)]
    im = cv2.imread(f"/kaggle/working/oi/{i}.jpg"); H, Wd = im.shape[:2]
    return im, np.stack([g.XMin * Wd, g.YMin * H, g.XMax * Wd, g.YMax * H], 1)

def sky_frac(im):
    """Share of the top half of the photo that is blue sky."""
    hsv = cv2.cvtColor(im[: im.shape[0] // 2], cv2.COLOR_BGR2HSV)
    h, s, v = hsv[..., 0], hsv[..., 1], hsv[..., 2]
    blue = (h >= 90) & (h <= 130) & (s > 40) & (v > 90)
    return float(blue.mean())
SKY = 0.4
used = set()   # never show the same photo twice

def show(im, dets, caption, seconds):
    c, s, ox, oy = fit(im)
    for d in dets:
        x1, y1, x2, y2 = (d["box"] * s + [ox, oy, ox, oy]).astype(int)
        col = RED if d["unknown"] else GREEN
        cv2.rectangle(c, (x1, y1), (x2, y2), col, 3)
        txt = f'UNKNOWN (looks like: {d["hint"]})' if d["unknown"] else f'{d["label"]} {d["conf"]:.2f}'
        label(c, txt, x1, y1 - 4, col, 0.7)
    banner(c, caption)
    for _ in range(int(seconds * FPS)):
        writer.write(c)

In [ ]:
# 5. Part 2: known classes. Keep photos where the model labels the main object correctly and adds no UNKNOWN.
card(["Part 2: the 6 known classes",
      "Photos from Open Images, a different source from the training data",
      "Green = known class + confidence"], 3)
shown = collections.Counter()
for c, name in KNOWN_OI.items():
    for i in pick.get(c, []):
        if i not in got or shown[name] >= 2: continue
        im, g = gt_boxes(i, c)
        if i in used or sky_frac(im) < SKY: continue
        dets = detect(im)
        ok = [d for d in dets if not d["unknown"] and d["label"] == name]
        allb = np.array([d["box"] for d in dets]).reshape(-1, 4)
        clean = len(ok) == len(dets) and (iou(allb, g).max(1) >= 0.3).all()
        if ok and clean and iou(g, np.array([d["box"] for d in ok])).max() >= 0.5:
            show(im, dets, f"Known class: {name}  |  combined YOLO11s model", 1.8); shown[name] += 1; used.add(i)
print("Known shown:", dict(shown))

In [ ]:
# 6. Part 3: never-seen objects. Keep photos where the object is flagged UNKNOWN and nothing gets a known label.
card(["Part 3: things it was never trained on",
      "Balloons, rockets, missiles, airships",
      "Red = UNKNOWN flag from YOLO-World (instead of a wrong known label)"], 3)
shown_u = collections.Counter()
for c in UNK_OI:
    for i in pick.get(c, []):
        if i not in got or shown_u[c] >= 3: continue
        im, g = gt_boxes(i, c)
        if i in used or sky_frac(im) < SKY: continue
        dets = detect(im)
        u = [d for d in dets if d["unknown"]]
        if u and len(u) == len(dets) and iou(g, np.array([d["box"] for d in u])).max() >= 0.4:
            show(im, dets, f"Never-seen object ({c.lower()})  |  flagged UNKNOWN", 2.2); shown_u[c] += 1; used.add(i)
print("Unknown shown:", dict(shown_u))

In [ ]:
# 7. End card with the honest full-test-set numbers, then encode to H.264
card(["Results (full test sets)",
      "Known classes: mAP50-95 0.760 on MBAOD, 0.670 on AOD4 (one model, both datasets)",
      "Drones on new footage: AP 0.284 -> 0.623 after training on varied data",
      "UNKNOWN flag: catches 32% of never-seen objects at 0.2-0.4% false flags",
      "Tracking on Anti-UAV video: precision 0.91, recall 0.89"], 6)
writer.release()
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", "/kaggle/working/raw_demo.mp4", "-c:v", "libx264",
                "-pix_fmt", "yuv420p", "-crf", "23", "-movflags", "+faststart", "/kaggle/working/airspace_demo.mp4"], check=True)
# A small preview GIF of the first drone clip for the README
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-ss", "8", "-t", "6", "-i", "/kaggle/working/airspace_demo.mp4",
                "-vf", "fps=10,scale=640:-1:flags=lanczos", "/kaggle/working/airspace_demo_preview.gif"], check=True)
import shutil
shutil.rmtree("/kaggle/working/oi", ignore_errors=True)
for p in glob.glob("/kaggle/working/*.pt") + ["/kaggle/working/raw_demo.mp4"]:
    os.remove(p)
cap = cv2.VideoCapture("/kaggle/working/airspace_demo.mp4")
print("Length (s):", round(cap.get(cv2.CAP_PROP_FRAME_COUNT) / FPS, 1))
print({f: round(os.path.getsize(f"/kaggle/working/{f}") / 1e6, 1) for f in os.listdir("/kaggle/working")})